# Silver Layer Transformation

This notebook transforms the Bronze-layer datasets into cleaned, standardized, and enriched datasets for downstream analysis.

In [0]:
airlines_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airlines')
airports_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airports')
flights_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.flights')
cancellation_codes_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.cancellation_codes')

In [0]:
from pyspark.sql import functions as F

## 1. Standardize Data Types

Convert columns to appropriate data types to ensure consistency and support further transformations.

In [0]:
# Review schema and sample records for each Bronze table to identify columns that require data type standardization
airlines_df.printSchema()
display(airlines_df.limit(5))

airports_df.printSchema()
display(airports_df.limit(5))

flights_df.printSchema()
display(flights_df.limit(5))

cancellation_codes_df.printSchema()
display(cancellation_codes_df.limit(5))

In [0]:
flights_df = flights_df.withColumn(
    'FLIGHT_NUMBER', F.col('FLIGHT_NUMBER').cast('string')
).withColumn(
    'DIVERTED', F.col('DIVERTED').cast('boolean')
).withColumn(
    'CANCELLED', F.col('CANCELLED').cast('boolean')
)

## 2. Handle missing values

Investigate missing values to determine whether they require correction or represent structural nulls.

In [0]:
# Investigate nulls identified during initial profiling
display(airports_df.filter(
    F.col('LATITUDE').isNull()
))

### Missing Airport Coordinates

Three airports contain missing latitude and longitude values. Since the affected airports can be identified by their IATA codes, their coordinates were manually referenced from Google Maps and used to fill the missing values.

In [0]:
missing_airport_coordinates = [
    ('ECP', 30.3582, -85.7956),
    ('PBG', 44.65094, -73.46814),
    ('UST', 29.959, -81.340)
]

# Create a reference dataframe
airports_ref = spark.createDataFrame(missing_airport_coordinates, schema=['IATA_CODE', 'LAT_REF', 'LON_REF'])

# Join reference coordinates and fill missing values
airports_df = airports_df.join(
                    airports_ref,
                    on='IATA_CODE',
                    how='left'
                ).withColumn(
                    'LATITUDE',
                    F.coalesce(F.col('LATITUDE'), F.col('LAT_REF'))
                ).withColumn(
                    'LONGITUDE',
                    F.coalesce(F.col('LONGITUDE'), F.col('LON_REF'))
                ).drop(
                    'LAT_REF',
                    'LON_REF'
                )

In [0]:
# Count null values in each column of the flights DataFrame
display(
    flights_df.select(
        [
            F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}')
            for column in flights_df.columns
        ]
    )
)

In [0]:
# Create FLIGHT_DATE to support the investigation of missing SCHEDULED_TIME values
flights_df = flights_df.withColumn(
    'FLIGHT_DATE',
    F.make_date(
        F.col('YEAR'),
        F.col('MONTH'),
        F.col('DAY')
    )
)

In [0]:
# Inspect records with missing SCHEDULED_TIME
display(
        flights_df.filter(
        F.col('SCHEDULED_TIME').isNull()
    ).select(
        'FLIGHT_DATE',
        'ORIGIN_AIRPORT',
        'DESTINATION_AIRPORT',
        'SCHEDULED_DEPARTURE',
        'SCHEDULED_ARRIVAL',
        'SCHEDULED_TIME'
    )
)

In [0]:
# Check if comparable flights exist to determine the missing SCHEDULED_TIME values
display(
    flights_df
    .filter(F.col('SCHEDULED_TIME').isNull())
    .alias('missing')
    .join(flights_df.alias('reference'),
            (
                (F.col('missing.ORIGIN_AIRPORT') == F.col('reference.ORIGIN_AIRPORT')) &
                (F.col('missing.DESTINATION_AIRPORT') == F.col('reference.DESTINATION_AIRPORT')) &
                (F.col('missing.SCHEDULED_DEPARTURE') == F.col('reference.SCHEDULED_DEPARTURE')) &
                (F.col('missing.SCHEDULED_ARRIVAL') == F.col('reference.SCHEDULED_ARRIVAL')) &
                F.col('reference.SCHEDULED_TIME').isNotNull()
            ),
    'inner')
    .select(
        F.col('missing.FLIGHT_DATE').alias('MISSING_FLIGHT_DATE'),
        F.col('reference.FLIGHT_DATE').alias('REFERENCE_FLIGHT_DATE'),
        'missing.ORIGIN_AIRPORT',
        'missing.DESTINATION_AIRPORT',
        'missing.SCHEDULED_DEPARTURE',
        'missing.SCHEDULED_ARRIVAL',
        F.col('reference.SCHEDULED_TIME').alias('REFERENCE_SCHEDULED_TIME')
    )
    .orderBy(
        'MISSING_FLIGHT_DATE', 'REFERENCE_FLIGHT_DATE'
    )
)

### Impute Missing Scheduled Times

Since comparable flights exist for the records with missing `SCHEDULED_TIME` and their `SCHEDULED_TIME` values are consistent, these values can be used to impute the missing entries.

In [0]:
# Create reference values for the missing SCHEDULED_TIME records
missing_scheduled_time = [
    ('2015-02-01', 172),
    ('2015-02-10', 172),
    ('2015-04-20', 178),
    ('2015-04-26', 111),
    ('2015-05-09', 130),
    ('2015-05-10', 113)
]

scheduled_time_reference = spark.createDataFrame(
    missing_scheduled_time, 
    ['FLIGHT_DATE', 'SCHEDULED_TIME_REF']
)

# Fill missing SCHEDULED_TIME values using the reference mapping
flights_df = flights_df.join(
    scheduled_time_reference,
    on='FLIGHT_DATE',
    how='left'
).withColumn(
    'SCHEDULED_TIME',
    F.coalesce('SCHEDULED_TIME', 'SCHEDULED_TIME_REF')
).drop(
    'SCHEDULED_TIME_REF'
)

Since many null values in the `flights` DataFrame occur in operational columns, investigate whether they represent genuinely missing data or are structurally expected based on the flight's operational status.

In [0]:
# Count flights by cancellation status

display(
    flights_df.groupBy(
        F.col('CANCELLED').alias('IS_CANCELLED')
    ).agg(
        F.count('CANCELLED').alias('COUNT')
    )
)

In [0]:
# For cancelled flights, how many rows per column are null
cancelled_flights = flights_df.filter(
    F.col('CANCELLED') == 'true'
)

display(
    cancelled_flights.select(
        [
            F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}')
            for column in cancelled_flights.columns
        ]
    )
)

In [0]:
# Of 89,884 cancelled flights, 86,153 have missing DEPARTURE_TIME and DEPARTURE_DELAY
# Inspect cancelled flights with departure information

display(cancelled_flights.filter(
    F.col('DEPARTURE_TIME').isNotNull() |
    F.col('DEPARTURE_DELAY').isNotNull())
)

# Check whether DEPARTURE_TIME and DEPARTURE_DELAY are populated for the same cancelled flights
display(
    cancelled_flights.filter(
        (F.col('DEPARTURE_TIME').isNotNull() &
        F.col('DEPARTURE_DELAY').isNull()) |
        (F.col('DEPARTURE_TIME').isNull() &
        F.col('DEPARTURE_DELAY').isNotNull())
))

In [0]:
# Inspect cancelled flights with TAXI_OUT or WHEELS_OFF information
display(
    cancelled_flights.filter(
        (F.col('TAXI_OUT').isNotNull()) |
        (F.col('WHEELS_OFF').isNotNull())
    )
)

# Check whether TAXI_OUT and WHEELS_OFF are populated for the same cancelled flights
display(cancelled_flights.filter(
    (F.col('TAXI_OUT').isNotNull() & F.col('WHEELS_OFF').isNull()) |
    (F.col('TAXI_OUT').isNull() & F.col('WHEELS_OFF').isNotNull())
))


**Conclusions so far:**
- Cancelled flights do not necessarily mean the aircraft never left the gate.
- Some cancelled flights have `DEPARTURE_TIME`, indicating that they departed the gate before cancellation.
- Some cancelled flights progressed further and have `TAXI_OUT` and `WHEELS_OFF` values.
- `WHEELS_OFF` is the furthest populated operational stage observed among cancelled flights.
- Cancelled flights with `WHEELS_OFF` have null values for subsequent operational columns such as `AIR_TIME`, `WHEELS_ON`, `TAXI_IN`, `ARRIVAL_TIME`, etc.

**BTS Documentation:**
According to the Bureau of Transportation Statistics, a flight may still be classified as **cancelled after wheels-off**. These cases are referred to as **fly returns**, where the aircraft takes off but subsequently returns.
- Therefore, null values in later operational columns for cancelled flights can be structurally expected and should not automatically be treated as missing errors.
- In addition, cancelled flights which does not already have an aircraft assigned prior to cancellation should record the flight's `TAIL_NUMBER` as null.
- These structural nulls are retained rather than imputed, as they reflect the operational status of the flight rather than a data quality issue.

In [0]:
# Inspect diverted flights
diverted_flights = flights_df.filter(
    F.col('DIVERTED') == 'true'
)

print('Diverted flights:', diverted_flights.count())
display(diverted_flights)

In [0]:
# Count null values in each column for diverted flights

display(
    diverted_flights.select(
        [
            F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}')
            for column in diverted_flights.columns
        ]
    )
)

In [0]:
# Since WHEELS_ON, TAXI_IN, and ARRIVAL_TIME have equal null counts, check whether their null patterns are identical
diverted_flights.filter(
    (F.col("WHEELS_ON").isNull() != F.col("TAXI_IN").isNull()) |
    (F.col("WHEELS_ON").isNull() != F.col("ARRIVAL_TIME").isNull())
).count()

In [0]:
# Check for records that both diverted and cancelled

flights_df.filter(
    (F.col('DIVERTED') == 'true') & (F.col('CANCELLED') == 'true') 
).count()


**Conclusions:**

The null values observed in the operational fields of diverted flights are structural nulls rather than missing data errors.

`ELAPSED_TIME`, `AIR_TIME`, and `ARRIVAL_DELAY` are null for all diverted flights in the dataset. Meanwhile, `WHEELS_ON`, `TAXI_IN`, `ARRIVAL_TIME`, and other related fields are populated for approximately 80% of diverted flights and are null for the remaining ~20%. The null values in these three columns were also confirmed to occur on the same records.

These null values represent operational information that is unavailable or not applicable due to the flight being diverted. Imputing them would introduce artificial information into the dataset. Therefore, they will be retained.

**BTS Documentation:**

BTS distinguishes between diverted flights that eventually reach their scheduled destination and those that do not. For diverted flights that do not reach the scheduled destination, standard arrival-related fields may be left blank. Diverted flights that eventually reach their scheduled destination may still contain values for fields such as `WHEELS_ON`, `TAXI_IN`, and `ARRIVAL_TIME`.

This behavior is consistent with the observed pattern in the dataset, where approximately 80% of diverted flights contain `WHEELS_ON`, `TAXI_IN`, and `ARRIVAL_TIME`, while the remaining ~20% have these fields as null.

In [0]:
# Inspect normal flights
normal_flights = flights_df.filter(
    (F.col('DIVERTED') == 'false') & (F.col('CANCELLED') == 'false')
)

print('Normal flights count: ', normal_flights.count())
display(normal_flights)

In [0]:
# Investigate nulls on normal_flights 

display(
    normal_flights.select(
        [F.count(
            F.when(
                F.col(column).isNull(), 1
            )
        ).alias(f'missing_{column}')
        for column in normal_flights.columns]
    )
)

In [0]:
# Investigate non-null values on the *_DELAY columns

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNotNull() |
        F.col('SECURITY_DELAY').isNotNull() |
        F.col('AIRLINE_DELAY').isNotNull() |
        F.col('LATE_AIRCRAFT_DELAY').isNotNull() |
        F.col('WEATHER_DELAY').isNotNull()
    )
)

In [0]:
# AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, and WEATHER_DELAY have the exact same amount of null values
# Additionally, these columns can can have a value that is 0
# Investigate if the null values are from the same records

display(
    normal_flights.filter(
        (
            F.col('AIR_SYSTEM_DELAY').isNull() |
            F.col('SECURITY_DELAY').isNull() |
            F.col('AIRLINE_DELAY').isNull() |
            F.col('LATE_AIRCRAFT_DELAY').isNull() |
            F.col('WEATHER_DELAY').isNull()
        )
        &
        (
            F.col('AIR_SYSTEM_DELAY').isNotNull() |
            F.col('SECURITY_DELAY').isNotNull() |
            F.col('AIRLINE_DELAY').isNotNull() |
            F.col('LATE_AIRCRAFT_DELAY').isNotNull() |
            F.col('WEATHER_DELAY').isNotNull()
        )
    )
)




In [0]:
# Since null values FROM *_DELAY columns are all from the same records, investigate these records

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNull()
    ).limit(30)
)

In [0]:
# Compare departure and arrival delay ranges based on whether AIR_SYSTEM_DELAY is populated

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNull()
    ).select(
        F.min('DEPARTURE_DELAY').alias('MIN DEPT DELAY'),
        F.max('DEPARTURE_DELAY').alias('MAX DEPT DELAY'),
        F.min('ARRIVAL_DELAY').alias('MIN ARR DELAY'),
        F.max('ARRIVAL_DELAY').alias('MAX ARR DELAY')
    )
)

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNotNull()
    ).select(
        F.min('DEPARTURE_DELAY').alias('MIN DEPT DELAY'),
        F.max('DEPARTURE_DELAY').alias('MAX DEPT DELAY'),
        F.min('ARRIVAL_DELAY').alias('MIN ARR DELAY'),
        F.max('ARRIVAL_DELAY').alias('MAX ARR DELAY')
    )
)

In [0]:
# Validate the observed 15-minute ARRIVAL_DELAY boundary for delay-cause reporting

display(
    normal_flights.filter(
        (F.col('AIR_SYSTEM_DELAY').isNull()) & (F.col('ARRIVAL_DELAY') >= 15)
    )
)

display(
    normal_flights.filter(
        (F.col('AIR_SYSTEM_DELAY').isNotNull()) & (F.col('ARRIVAL_DELAY') < 15)
    )
)


**Conclusions:**
- `AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `WEATHER_DELAY`, `AIRLINE_DELAY`, and `LATE_AIRCRAFT_DELAY` are null when `ARRIVAL_DELAY` is less than 15 minutes.
- For flights delayed 15 minutes or more, all five delay-cause columns are populated, including `0` when no delay is attributed to a particular cause.

**BTS Documentation:**
- On BTS reporting rules, causal delay data are required only for flights arriving at least 15 minutes late.
- Therefore, nulls in these columns for flights with `ARRIVAL_DELAY < 15` are structural and should not be imputed.

## 3. Check for duplicate records
Identify duplicate records and remove them where necessary.

In [0]:
print('Airlines DataFrame')
print('Duplicate records:', airlines_df.count() - airlines_df.dropDuplicates().count())

print('\nAirports DataFrame')
print('Duplicate records:', airports_df.count() - airports_df.dropDuplicates().count())

print('\nCancellation Codes DataFrame')
print('Duplicate records:', cancellation_codes_df.count() - cancellation_codes_df.dropDuplicates().count())

print('\nFlights DataFrame')
print('Duplicate records:', flights_df.count() - flights_df.dropDuplicates().count())

## 4. Standardize inconsistent values and identifiers
Identify and standardize inconsistent values and coding formats to ensure that equivalent values are represented consistently.

In [0]:
# Initial profiling showed that ORIGIN_AIRPORT contains inconsistent identifier formats
display(
    flights_df.select('ORIGIN_AIRPORT')
    .distinct()
    .orderBy('ORIGIN_AIRPORT')
)

In [0]:
# Check whether airport identifiers use formats other than 3-letter codes or 5-digit DOT codes

display(
    flights_df.groupBy(
        F.when(F.col('ORIGIN_AIRPORT')
               .rlike('^[A-Z]{3}$'),
               'IATA_CODE')
        .when(F.col('ORIGIN_AIRPORT')
              .rlike('^[0-9]{5}$'),
              'DOT_CODE')
        .otherwise('UNKOWN_FORMAT')
        .alias('ORIGIN AIRPORT CODE TYPE')
    ).count()
)

display(
    flights_df.groupBy(
        F.when(F.col('DESTINATION_AIRPORT')
               .rlike('^[A-Z]{3}$'),
               'IATA_CODE')
        .when(F.col('DESTINATION_AIRPORT')
              .rlike('^[0-9]{5}$'),
              'DOT_CODE')
        .otherwise('UNKOWN_FORMAT')
        .alias('DESTINATION AIRPORT CODE TYPE')
    ).count()
)

### Airport Identifier Reference Data

The airport mapping dataset was obtained from the U.S. Department of Transportation's Bureau of Transportation Statistics (BTS) TranStats database and is used as a reference for mapping 5-digit DOT airport IDs to airport codes.

In [0]:
airport_mapping_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv('/Volumes/flight-delay-analytics-catalog/bronze/raw_files/airport_id_mapping.csv')
)

display(airport_mapping_df)

In [0]:
# Check for duplicate AIRPORT_ID values in the airport mapping data

display(
    airport_mapping_df.groupBy('AIRPORT_ID')
    .count()
    .filter(F.col('count') > 1)
    .orderBy('count', ascending=False))

In [0]:
display(airport_mapping_df.filter(
    F.col('AIRPORT_ID') == '11881'
))

In [0]:
# Inspect schema of airport_mapping_df
airport_mapping_df.printSchema()

In [0]:
# Standardize airport ID and date column data types
airport_mapping_df = (
    airport_mapping_df
    .withColumn(
        'AIRPORT_ID',
        F.col('AIRPORT_ID').cast('string')
    )
    .withColumn(
        'AIRPORT_START_DATE',
        F.try_to_timestamp(
            F.col('AIRPORT_START_DATE'),
            F.lit('M/d/yyyy h:mm:ss a')
        )
    )
    .withColumn(
        'AIRPORT_THRU_DATE',
        F.try_to_timestamp(
            F.col('AIRPORT_THRU_DATE'),
            F.lit('M/d/yyyy h:mm:ss a')
        )
    )
)

In [0]:
# Select airport mapping records valid at the October 2015 identifier switch
airport_mapping_2015_df = airport_mapping_df.filter(
    (F.col('AIRPORT_START_DATE') <= F.lit('2015-10-01')) &
    (
        F.col('AIRPORT_THRU_DATE').isNull() |
        (F.col('AIRPORT_THRU_DATE') >= F.lit('2015-10-01'))
    )
)

display(airport_mapping_2015_df)

In [0]:
# Verify that all AIRPORT_ID values are unique after filtering
display(
    airport_mapping_2015_df.groupBy('AIRPORT_ID')
    .count()
    .filter(F.col('count') > 1)
    .orderBy('count', ascending=False))

In [0]:
# Select only airport identifiers needed for mapping
selected_airport_mapping_df = airport_mapping_2015_df.select(
    F.col('AIRPORT_ID').alias('NEW_AIRPORT_ID'),
    F.col('AIRPORT').alias('NEW_AIRPORT_CODE')
)

display(selected_airport_mapping_df)

In [0]:
# Map 5-digit DOT origin and destination airport IDs to their corresponding 3-letter airport codes

flights_df = flights_df.join(
                selected_airport_mapping_df,
                flights_df['ORIGIN_AIRPORT'] == selected_airport_mapping_df['NEW_AIRPORT_ID'],
                'left'
            ).withColumn(
                'ORIGIN_AIRPORT_STANDARDIZED',
                F.coalesce(
                    F.col('NEW_AIRPORT_CODE'),
                    F.col('ORIGIN_AIRPORT')
                )
            ).drop(
                'NEW_AIRPORT_ID',
                'NEW_AIRPORT_CODE'
            ).join(
                selected_airport_mapping_df,
                flights_df['DESTINATION_AIRPORT'] == selected_airport_mapping_df['NEW_AIRPORT_ID'],
                'left'
            ).withColumn(
                'DESTINATION_AIRPORT_STANDARDIZED',
                F.coalesce(
                    F.col('NEW_AIRPORT_CODE'),
                    F.col('DESTINATION_AIRPORT')
                )
            ).drop(
                'NEW_AIRPORT_ID',
                'NEW_AIRPORT_CODE'
            )

display(flights_df)

## 5. Create derived columns
Create additional columns that support analysis and can be used in the Gold layer.

Up until this point, the date and time columns in `flights_df` are still stored as integers. Creating proper datetime columns will make these values easier to work with and support time-based analysis in the Gold layer.

In [0]:
# Create scheduled departure datetime from FLIGHT_DATE and SCHEDULED_DEPARTURE
flights_df = flights_df.withColumn(
    'SCHEDULED_DEPARTURE_DT',
    F.to_timestamp_ntz(
        F.concat_ws(
            ' ',
            F.col('FLIGHT_DATE'),
            F.lpad(
                F.col('SCHEDULED_DEPARTURE').cast('string'),
                4,
                '0'
            )
        ),
        F.lit('yyyy-MM-dd HHmm')
    )
)

In [0]:
# Inspect flights where DEPARTURE_TIME occurred the day after SCHEDULED_DEPARTURE
display(
    flights_df.filter(
        (
            (F.floor(F.col('SCHEDULED_DEPARTURE') / 100) * 60)
            + (F.col('SCHEDULED_DEPARTURE') % 100)
            + F.col('DEPARTURE_DELAY')
        ) >= 1440
    )
)

In [0]:
# Derive DEPARTURE_TIME_DT and WHEELS_OFF_DT while accounting for date rollover
flights_df = flights_df.withColumn(
                'DEPARTURE_TIME_DT',
                F.expr(
                    "SCHEDULED_DEPARTURE_DT + INTERVAL 1 MINUTE * DEPARTURE_DELAY"
                )
            ).withColumn(
                'WHEELS_OFF_DT',
                F.expr(
                    "DEPARTURE_TIME_DT + INTERVAL 1 MINUTE * TAXI_OUT"
                )
            )

Up until this point, datetime columns have been derived by adding integer-based duration values to existing timestamps. However, `WHEELS_ON` and subsequent operational times are recorded in the destination airport's local time. Therefore, the time zone difference between the origin and destination airports must be taken into account when deriving these datetime columns.

In [0]:
from timezonefinder import TimezoneFinder

In [0]:
# Derive airport time zones from latitude and longitude coordinates
from pyspark.sql.types import StringType

def get_timezone(lat, lon):
    from timezonefinder import TimezoneFinder
    tf_local = TimezoneFinder()
    return tf_local.timezone_at(lat=lat, lng=lon)

timezone_udf = F.udf(get_timezone, StringType())

airports_df = airports_df.withColumn(
    'TIME_ZONE',
    timezone_udf(
        F.col('LATITUDE'),
        F.col('LONGITUDE')
    )
)

display(airports_df.limit(20))

In [0]:
# Add origin and destination airport time zones to flights_df
timezone_lookup_df = airports_df.select(
    'IATA_CODE',
    'TIME_ZONE'
)

flights_df = flights_df.join(
    F.broadcast(timezone_lookup_df),
    flights_df['ORIGIN_AIRPORT_STANDARDIZED'] == timezone_lookup_df['IATA_CODE'],
    how='left'
).withColumnRenamed(
    'TIME_ZONE', 'ORIGIN_AIRPORT_TZ'
).drop('IATA_CODE')

flights_df = flights_df.join(
    F.broadcast(timezone_lookup_df),
    flights_df['DESTINATION_AIRPORT_STANDARDIZED'] == timezone_lookup_df['IATA_CODE'],
    'left'
).withColumnRenamed(
    'TIME_ZONE', 'DESTINATION_AIRPORT_TZ'
).drop('IATA_CODE')

display(flights_df.limit(10))

In [0]:
# Persist the flights DataFrame as a Delta table to avoid recomputing its full transformation lineage
(
    flights_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('`flight-delay-analytics-catalog`.silver.flights')
)

flights_df = spark.table('`flight-delay-analytics-catalog`.silver.flights')

In [0]:
flights_df = spark.table('`flight-delay-analytics-catalog`.silver.flights')
airports_df = spark.table('`flight-delay-analytics-catalog`.bronze.airports')

from pyspark.sql import functions as F

In [0]:
display(flights_df.filter(
    F.col('ORIGIN_AIRPORT_TZ').isNull() | F.col('DESTINATION_AIRPORT_TZ').isNull(
)))

In [0]:
# Convert WHEELS_OFF_DT from origin local time to UTC for WHEELS_ON derivation
flights_df = flights_df.withColumn(
    'WHEELS_OFF_UTC',
    F.expr(
        "convert_timezone(ORIGIN_AIRPORT_TZ, 'UTC', WHEELS_OFF_DT)"
    )
).withColumn(
    'WHEELS_ON_UTC',
    F.expr('WHEELS_OFF_UTC + INTERVAL 1 MINUTE * AIR_TIME')
).withColumn(
    'WHEELS_ON_DT',
    F.expr(
        "convert_timezone('UTC', DESTINATION_AIRPORT_TZ, WHEELS_ON_UTC)"
    )
)

display(flights_df.limit(3))

In [0]:
# Display and inspect rows where WHEELS_ON and derived WHEELS_ON_DT do not match 

mismatched_wheels_on = flights_df.filter(
    F.col('WHEELS_ON').isNotNull() &
    (
        F.lpad(
            F.col('WHEELS_ON').cast('string'),
            4,
            '0'
        ) !=
        F.date_format(
            F.col('WHEELS_ON_DT'),
            'HHmm'
        )
    )
).select(
    'FLIGHT_DATE',
    'ORIGIN_AIRPORT_STANDARDIZED',
    'DESTINATION_AIRPORT_STANDARDIZED',
    'ORIGIN_AIRPORT_TZ',
    'DESTINATION_AIRPORT_TZ',
    'WHEELS_OFF',
    'WHEELS_OFF_DT',
    'WHEELS_OFF_UTC',
    'AIR_TIME',
    'WHEELS_ON_UTC',
    'WHEELS_ON',
    'WHEELS_ON_DT'
)

display(mismatched_wheels_on)


### Investigate `WHEELS_ON` Mismatches

Initial inspection of the mismatched records showed that many were caused by a difference in midnight representation. The raw `WHEELS_ON` column represents midnight as `2400`, while the derived `WHEELS_ON_DT` timestamp represents midnight as `00:00`.

Since these values represent the same time, records where `WHEELS_ON = 2400` are excluded from further mismatch investigation.

In [0]:
mismatched_wheels_on = mismatched_wheels_on.filter(
    F.col('WHEELS_ON') != 2400
)

display(mismatched_wheels_on)

In [0]:
# Check whether the remaining mismatches are concentrated among specific airport routes or time zones
display(
    mismatched_wheels_on.groupBy(
        'ORIGIN_AIRPORT_STANDARDIZED',
        'DESTINATION_AIRPORT_STANDARDIZED',
        'ORIGIN_AIRPORT_TZ',
        'DESTINATION_AIRPORT_TZ'
    )
    .count()
    .orderBy('count', ascending=False)
)

### Remaining Mismatch Pattern

After excluding the `2400` midnight cases, the remaining mismatches were grouped by route and airport time zone to identify potential patterns.

The results show that a large proportion of the remaining mismatches occur on flights between GUM and HNL. Since these routes involve time zone conversion, the assigned airport time zones should be investigated further.

In [0]:
display(
    mismatched_wheels_on.filter(
        (F.col('ORIGIN_AIRPORT_STANDARDIZED') == 'GUM') &
        (F.col('DESTINATION_AIRPORT_STANDARDIZED') == 'HNL')
    )
    .select(
        'WHEELS_OFF', 'WHEELS_OFF_DT', 'AIR_TIME', 'WHEELS_OFF_UTC',
        'WHEELS_ON_UTC', 'WHEELS_ON', 'WHEELS_ON_DT'
    )
)

In [0]:
display(
    airports_df.filter(
        F.col('IATA_CODE') == 'GUM'
    ).select(
        'IATA_CODE',
        'LATITUDE',
        'LONGITUDE'#,
        #'TIME_ZONE'
    )
)

In [0]:
mismatched_airports = (
    mismatched_wheels_on
    .select(
        F.col('ORIGIN_AIRPORT_STANDARDIZED').alias('IATA_CODE')
    )
    .union(
        mismatched_wheels_on.select(
            F.col('DESTINATION_AIRPORT_STANDARDIZED').alias('IATA_CODE')
        )
    )
    .distinct()
)

display(
    mismatched_airports
    .join(
        airports_df.select(
            'IATA_CODE',
            'LATITUDE',
            'LONGITUDE'#,
            #'TIME_ZONE'
        ),
        on='IATA_CODE',
        how='left'
    )
    .orderBy('IATA_CODE')
)

In [0]:
display(
    mismatched_wheels_on.filter(
        (F.col('ORIGIN_AIRPORT_STANDARDIZED') == 'SEA') &
        (F.col('DESTINATION_AIRPORT_STANDARDIZED') == 'ANC')
    )
)

In [0]:
display(
    airports_df.filter(
        (F.col('IATA_CODE') == 'GUM') |
        (F.col('IATA_CODE') == 'ANC') |
        (F.col('IATA_CODE') == 'SEA')
    )
)

In [0]:
display(
    mismatched_wheels_on.filter(
        (F.col('ORIGIN_AIRPORT_STANDARDIZED') == 'ANC') &
        (F.col('DESTINATION_AIRPORT_STANDARDIZED') == 'SEA')
    ).select(
        'FLIGHT_DATE',
        'ORIGIN_AIRPORT_TZ',
        'DESTINATION_AIRPORT_TZ',
        'WHEELS_OFF',
        'WHEELS_OFF_DT',
        'WHEELS_OFF_UTC',
        'AIR_TIME',
        'WHEELS_ON_UTC',
        'WHEELS_ON',
        'WHEELS_ON_DT'
    )
)

In [0]:
display(
    flights_df.filter(
        (F.col('ORIGIN_AIRPORT_STANDARDIZED') == 'ANC') &
        (F.col('DESTINATION_AIRPORT_STANDARDIZED') == 'SEA') &
        (F.col('WHEELS_ON').isNotNull()) &
        (F.col('WHEELS_ON') != 2400) &
        (
            F.lpad(
                F.col('WHEELS_ON').cast('string'),
                4,
                '0'
            ) !=
            F.date_format(
                F.col('WHEELS_ON_DT'),
                'HHmm'
            )
        )
    ).select(
        'FLIGHT_DATE',
        'SCHEDULED_DEPARTURE_DT',
        'DEPARTURE_TIME_DT',
        'WHEELS_OFF_DT',
        'ORIGIN_AIRPORT_TZ',
        'WHEELS_OFF_UTC',
        'AIR_TIME',
        'WHEELS_ON_UTC',
        'DESTINATION_AIRPORT_TZ',
        'WHEELS_ON_DT',
        'WHEELS_ON'
    )
)